# OvisOCR3 — structured document JSON on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/OvisOCR3-structured-json.ipynb)

[`ATH-MaaS/OvisOCR3`](https://huggingface.co/ATH-MaaS/OvisOCR3) is a unified **0.85B** (853M) multimodal OCR model, Apache 2.0, built on Qwen3.5-0.8B (about 1.7 GB of fp16 weights). Task-specific prompts pick the output contract: invoice fields as JSON, a page as a JSON array of regions, Markdown with HTML tables, or a two-page table rebuild. That is the point of this notebook — **prompt-driven structured JSON**, not a dump of grounded OCR text.

**Licence:** Apache 2.0.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has 16 GB of VRAM, no native bfloat16, and no FlashAttention 2. This notebook loads the checkpoint in **float16** with **SDPA** attention (eager if SDPA refuses the hybrid stack). Weights are about 1.7 GB; a page whose longest edge is 2048 px peaks at roughly 5–8 GB. That estimate is from the weight file plus a 2048 px page, not a measured Colab run.

**Not vLLM.** The model card's published recipe uses `vllm==0.22.1`. vLLM is not a free-T4 path here (install size, CUDA / FlashAttention). Transformers loads the same `Qwen3_5ForConditionalGeneration` checkpoint.

**Not in this notebook:** LightOnOCR's empty / `grounding` prompts, handwriting transcription, or a LoRA fine-tune.

**Links:** [model card](https://huggingface.co/ATH-MaaS/OvisOCR3) · [demo](https://huggingface.co/spaces/ATH-MaaS/OvisOCR3)


In [ ]:
%pip install -q "transformers==5.20.1" "pypdfium2==5.14.0" pillow

# OvisOCR3 is Qwen3_5ForConditionalGeneration, the same class as LightOnOCR-3 0.8B.
# transformers 5.20.1 is the pin those T4 notebooks already use (floor is 5.5.4).
# The card's install is vllm==0.22.1; this notebook stays on transformers so a T4 can load it.
# pypdfium2 renders the PDF page. huggingface_hub (already a transformers dep) fetches the official prompts/.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.


In [ ]:
import torch
import transformers
from packaging.version import Version

assert Version(transformers.__version__) >= Version("5.5.4"), (
    f"transformers {transformers.__version__} is loaded; OvisOCR3 needs 5.5.4+. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention 2. Keep PyTorch SDPA on the mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, minor = torch.cuda.get_device_capability()
DTYPE = torch.bfloat16 if major >= 8 else torch.float16
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
if major < 8:
    print("T4 path: float16 + SDPA/eager. bfloat16 and FlashAttention-2 are off.")


## 1. Load OvisOCR3

The Hub config is `Qwen3_5ForConditionalGeneration` with `AutoProcessor`. Thinking stays off: the card's chat template and the published vLLM example both pass `enable_thinking=False`. Sampling is greedy (`do_sample=False`, temperature 0), matching `generation_config.json`. The first run downloads about 1.7 GB.

The card's vLLM recipe allows up to `2880×2880` pixels per image. That is too many visual tokens for a comfortable T4 generate. After load, the processor is capped at `1280×1280` pixels (`min_pixels` stays at the card's `448×448`). Pages are also limited to a 2048 px longest edge before they reach the processor.


In [ ]:
from pathlib import Path

from huggingface_hub import hf_hub_download
from transformers import AutoProcessor, Qwen3_5ForConditionalGeneration

MODEL_ID = "ATH-MaaS/OvisOCR3"
MIN_PIXELS = 448 * 448
MAX_PIXELS = 1280 * 1280  # T4 cap; the card's vLLM example uses 2880 * 2880


def load_ocr(attn):
    return Qwen3_5ForConditionalGeneration.from_pretrained(
        MODEL_ID,
        dtype=DTYPE,
        attn_implementation=attn,
        device_map="auto",
    ).eval()


try:
    model = load_ocr("sdpa")
    ATTN = "sdpa"
except Exception as exc:
    print("sdpa load failed, retrying eager:", exc)
    model = load_ocr("eager")
    ATTN = "eager"

processor = AutoProcessor.from_pretrained(MODEL_ID)
image_proc = processor.image_processor
if hasattr(image_proc, "min_pixels"):
    image_proc.min_pixels = MIN_PIXELS
    image_proc.max_pixels = MAX_PIXELS
image_proc.size = {"shortest_edge": MIN_PIXELS, "longest_edge": MAX_PIXELS}

print("attention:", ATTN, "| dtype:", model.dtype)
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


## Helpers and official task prompts

Prompts are the files under [`prompts/`](https://huggingface.co/ATH-MaaS/OvisOCR3/tree/main/prompts) in the model repo. Placeholders are replaced with `str.replace`, not `str.format` — the templates contain JSON and LaTeX braces. `infer` does not repair JSON or strip fences; the card leaves raw model text unchanged.


In [ ]:
import io
import json
import re
import time

import requests
from IPython.display import HTML, display
from PIL import Image, ImageDraw, ImageFont

RECEIPT_URL = "https://huggingface.co/datasets/hf-internal-testing/fixtures_ocr/resolve/main/SROIE-receipt.jpeg"
PAPER_URL = "https://arxiv.org/pdf/1706.03762"
UA = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}


def load_prompt(name: str) -> str:
    path = hf_hub_download(MODEL_ID, filename=f"prompts/{name}.txt")
    return Path(path).read_text(encoding="utf-8")


def fetch_bytes(url, attempts=4):
    for attempt in range(attempts):
        response = requests.get(url, headers=UA, timeout=120)
        if response.status_code not in (429, 500, 502, 503, 504) or attempt == attempts - 1:
            response.raise_for_status()
            return response.content
        retry_after = response.headers.get("Retry-After", "")
        time.sleep(int(retry_after) if retry_after.isdigit() else 2 ** (attempt + 1))


def fetch_image(url):
    return Image.open(io.BytesIO(fetch_bytes(url))).convert("RGB")


def render_pdf_page(url, page_index, longest=2048):
    import pypdfium2 as pdfium

    pdf = pdfium.PdfDocument(fetch_bytes(url))
    image = pdf[page_index].render(scale=400 / 72).to_pil().convert("RGB")
    image.thumbnail((longest, longest))
    return image


def limit_edge(image, longest):
    image = image.copy()
    image.thumbnail((longest, longest))
    return image


def to_model(inputs):
    inputs = inputs.to(model.device)
    for key, value in list(inputs.items()):
        if hasattr(value, "is_floating_point") and value.is_floating_point():
            inputs[key] = value.to(dtype=model.dtype)
    return inputs


def infer(images, task_prompt, max_new_tokens=1024):
    if not 1 <= len(images) <= 2:
        raise ValueError("The published example accepts one or two images per request.")
    content = [{"type": "image", "image": image.convert("RGB")} for image in images]
    content.append({"type": "text", "text": task_prompt})
    inputs = processor.apply_chat_template(
        [{"role": "user", "content": content}],
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    )
    inputs = to_model(inputs)
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    generated = output_ids[0, inputs["input_ids"].shape[1] :]
    text = processor.decode(generated, skip_special_tokens=True)
    if torch.cuda.is_available():
        print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    return text


def show_json(text):
    print(text)
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError as exc:
        print("JSON parse failed (model can emit malformed JSON):", exc)
        return None
    print("\nparsed:")
    print(json.dumps(parsed, indent=2, ensure_ascii=False))
    return parsed


KIE_SCHEMA = load_prompt("kie_schema")
DOCUMENT_STRUCTURED = load_prompt("document_structured")
DOCUMENT_MARKDOWN = load_prompt("document_markdown")
CROSSPAGE_TABLE = load_prompt("crosspage_table")
print("loaded prompts:", "kie_schema, document_structured, document_markdown, crosspage_table")


## 2. Demo 1 — receipt / invoice → JSON fields

The SROIE receipt from the LightOnOCR notebook, but the prompt is OvisOCR3's schema-based KIE template (`prompts/kie_schema.txt`). The schema is an example of the desired JSON, not a formal JSON Schema document. Empty strings are placeholders. Asked fields: vendor, date, total, and line items.


In [ ]:
receipt = limit_edge(fetch_image(RECEIPT_URL), 2048)
print("receipt", receipt.size)
display(receipt)

schema = {
    "vendor": "",
    "date": "",
    "total": "",
    "line_items": [
        {
            "description": "",
            "quantity": "",
            "unit_price": "",
            "amount": "",
        }
    ],
}
schema_prompt = KIE_SCHEMA.replace("{schema_json}", json.dumps(schema, ensure_ascii=False))
print("prompt (truncated):")
print(schema_prompt[:280], "...")

receipt_json = infer([receipt], schema_prompt, max_new_tokens=768)
show_json(receipt_json)


## 3. Demo 2 — table page → structured reconstruction

Two reconstructions of the same capability, both prompt-selected:

1. **One page, structured JSON.** Page 8 of [Attention Is All You Need](https://arxiv.org/pdf/1706.03762) (arXiv:1706.03762) holds Table 2. `document_structured` returns a JSON array of `{bbox, category, content}` records; table `content` is an HTML `<table>`. Coordinates are integers in `[0, 1000)`.
2. **Two pages, official rebuild.** The card's cross-page recipe is two-stage: parse each page, then send both images plus their Markdown to `crosspage_table`. The demo uses two consecutive pages drawn here so the continuation is visible; a live multi-page PDF follows the same calls.


In [ ]:
table_page = render_pdf_page(PAPER_URL, page_index=7)  # printed page 8, Table 2
print("table page", table_page.size)
display(table_page)

structured = infer([table_page], DOCUMENT_STRUCTURED, max_new_tokens=1600)
parsed_page = show_json(structured)

if isinstance(parsed_page, list):
    tables = [el for el in parsed_page if isinstance(el, dict) and el.get("category") == "table"]
    print(f"{len(tables)} table region(s)")
    for el in tables:
        display(HTML(el.get("content") or ""))


### Cross-page table rebuild (two images)

Stage 1 parses each page with `document_markdown`. Stage 2 concatenates those Markdown blobs into `<INPUT_PAGE>` blocks and calls `crosspage_table` with **both** page images. The published contract is a JSON array of `{source_pages, html}` records, or `[{"source_pages":[],"html":"No cross-page table exists between these pages."}]` when nothing continues.


In [ ]:
def draw_table_page(title, subtitle, headers, rows, footer, size=(960, 1280)):
    image = Image.new("RGB", size, (250, 248, 242))
    draw = ImageDraw.Draw(image)
    font_pairs = (
        (
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
            "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        ),
        (
            "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
        ),
    )
    title_font = body_font = small_font = ImageFont.load_default()
    for bold_path, regular_path in font_pairs:
        try:
            title_font = ImageFont.truetype(bold_path, 36)
            body_font = ImageFont.truetype(regular_path, 22)
            small_font = ImageFont.truetype(regular_path, 18)
            break
        except OSError:
            continue

    draw.rectangle((40, 40, size[0] - 40, size[1] - 40), outline=(40, 40, 40), width=2)
    draw.text((64, 64), title, fill=(20, 20, 20), font=title_font)
    draw.text((64, 118), subtitle, fill=(80, 80, 80), font=small_font)

    col_x = [64, 360, 560, 720]
    y = 180
    for x, header in zip(col_x, headers):
        draw.text((x, y), header, fill=(20, 20, 20), font=body_font)
    y += 36
    draw.line((64, y, size[0] - 64, y), fill=(40, 40, 40), width=2)
    y += 16
    for row in rows:
        for x, cell in zip(col_x, row):
            draw.text((x, y), cell, fill=(20, 20, 20), font=body_font)
        y += 40
    draw.text((64, size[1] - 88), footer, fill=(80, 80, 80), font=small_font)
    return image


page1 = draw_table_page(
    "Harbor Supply Co.",
    "Stock transfer · 18 Mar 2026 · page 1 of 2",
    ["Item", "SKU", "Qty", "Amount"],
    [
        ("Cedar fence posts", "HSC-110", "40", "320.00"),
        ("Galvanised hinges", "HSC-204", "12", "84.00"),
        ("Deck screws 8x", "HSC-318", "6", "27.60"),
        ("Exterior stain 4L", "HSC-441", "8", "176.00"),
    ],
    "Continued on next page →",
)
page2 = draw_table_page(
    "Harbor Supply Co.",
    "Stock transfer · 18 Mar 2026 · page 2 of 2",
    ["Item", "SKU", "Qty", "Amount"],
    [
        ("Safety goggles", "HSC-502", "10", "45.00"),
        ("Work gloves L", "HSC-519", "20", "90.00"),
        ("Tarpaulin 4x6 m", "HSC-630", "3", "54.00"),
    ],
    "Subtotal 796.60   Tax 63.73   Total 860.33",
)

print("synthetic pages", page1.size, page2.size)
display(page1)
display(page2)

page_markdowns = []
for index, page in enumerate((page1, page2), start=1):
    raw = infer([page], DOCUMENT_MARKDOWN, max_new_tokens=700)
    markdown = re.sub(r"^```(?:markdown|html)?\s*\n|\n```\s*$", "", raw.strip(), flags=re.I)
    print(f"\n--- page {index} markdown ---")
    print(markdown)
    page_markdowns.append(markdown)

page_context = "\n\n".join(
    f'<INPUT_PAGE index="{index}">\n{markdown}\n</INPUT_PAGE>'
    for index, markdown in enumerate(page_markdowns, start=1)
)
cross_prompt = CROSSPAGE_TABLE.removesuffix("\n") + "\n\nInput Markdown:\n\n" + page_context
tables_json = infer([page1, page2], cross_prompt, max_new_tokens=900)
parsed_tables = show_json(tables_json)
if isinstance(parsed_tables, list):
    for record in parsed_tables:
        html = record.get("html") if isinstance(record, dict) else None
        if html and "<table" in html.lower():
            display(HTML(html))


## 4. Caveats — and how this differs from the LightOnOCR notebooks

**What OvisOCR3 is for.** A single 0.85B checkpoint switches tasks by prompt: open / field / schema KIE, structured or Markdown page parsing, figure captions, and a two-image table rebuild. The output you want is in the prompt file, not in a second model.

**What LightOnOCR-3 is for.** The [0.8B notebook](LightOnOCR-3-0.8B-document-ocr.ipynb) (and any 4B sibling) is **grounded OCR**: an empty prompt transcribes the page to Markdown; `grounding` adds a label and a 0–1000 box for every block. Charts come back as HTML of plotted values. It does not take a JSON schema or a two-page rebuild prompt. Use that notebook when you want text-plus-boxes. Use this one when you want invoice fields or a reconstructed table as JSON.

| | OvisOCR3 (this notebook) | LightOnOCR-3 0.8B / 4B |
| --- | --- | --- |
| Job | Prompt-selected structured output | Transcription + optional grounding boxes |
| Receipt | `kie_schema` → vendor / date / total / lines | `grounding` → markdown blocks + boxes |
| Table | JSON regions or a two-page HTML rebuild | Grounded blocks; table text, not a merge API |
| Runtime here | float16 + SDPA/eager on a free T4 | Same T4 recipe |

**T4 limits.** No bfloat16, no FlashAttention-2, no vLLM in this notebook. `max_pixels` is `1280×1280`, not the card's `2880×2880`. Two images in one request is the published maximum; a long table can still truncate (`max_new_tokens` is capped so the cell finishes). The card's 16384-token vLLM budget is not used.

**Outputs can be wrong.** The card says the model can emit incorrect, incomplete, repeated, or malformed JSON. Small or blurred text, dense layouts, and long tables are the usual failure modes. Cross-page rebuild quality depends on the first-stage Markdown and on page order. Validate JSON before you trust it; this notebook does not repair the raw string.

**Samples.** The SROIE receipt is the LightOnOCR fixture, so you can run both notebooks on the same image and compare grounded text with schema JSON. The Attention Is All You Need page is arXiv:1706.03762. The two-page stock list is drawn in the notebook so the continuation is known.
